# class TrendModeT(tp.NamedTuple)
趋势模式枚举类型

```python
class TrendModeT(tp.NamedTuple):
    Binary: int = 0
    BinaryCont: int = 1
    BinaryContSat: int = 2
    PctChange: int = 3
    PctChangeNorm: int = 4
```

## `Binary`
二进制趋势标签模式，用于基本的趋势方向分类
    
在局部极值点之间的区间内，根据趋势方向分配标签：下跌趋势为0，上涨趋势为1。

标签含义：
- 0：下跌趋势（从波峰到波谷的区间）
- 1：上涨趋势（从波谷到波峰的区间）

## `BinaryCont`
连续二进制趋势标签模式，在每个极值区间内将价格标准化为0-1的连续值

标签含义：
- 接近 0 的值：当前价格接近区间最高点，预期将下跌
- 接近 1 的值：当前价格接近区间最低点，预期将上涨
- 中间值：表示上涨或下跌的程度

计算原理：在每个极值区间内，使用公式
$$label = 1 - \frac{{current\_price{\rm{ }} - {\rm{ }}min\_price}}{{max\_price{\rm{ }} - {\rm{ }}min\_price}}$$

## `BinaryContSat`
饱和连续趋势标签模式，在连续标签的基础上增加了饱和处理机制

当价格变化超过设定的阈值时，标签会被设置为饱和值（0或1），否则使用连续的插值。

标签含义：
- 0：强烈的下跌信号（饱和状态）
- 1：强烈的上涨信号（饱和状态）
- 中间值: 线性插值的趋势强度

饱和机制
- 已知
  - 价格序列 ${p_1}, \cdots ,{p_T}$ 和极值点标记 ${E_1}, \cdots ,{E_T}$
  - 上涨阈值 $pos_{th}$ 和下跌阈值 $neg_{th}$
- 对于上涨区间
  - 如果
    - $${p_t} \le \frac{{{p_{\max }}}}{{1 + pos_{th}}}$$ 
    - $${p_{\max }} \ge {p_{\min }}\left( {1 + pos_{th}} \right)$$
    则认为 $L_t=1$，即极端看涨
  - 否则 
    - $${L_t} = 1 - \frac{{{P_t} - \frac{{\max }}{{1 + po{s_{th}}}}}}{{\max  - \frac{{\max }}{{1 + po{s_{th}}}}}}$$
    $p_t$ 越接近于 $\frac{{\max }}{{1 + po{s_{th}}}}$，$L_t$ 越接近于 1，即越看涨；否则 $p_t$ 越接近于 $p_{max}$，$L_t$ 越接近于 0，看涨减弱
- 对于下跌区间
  - 如果
    - $${p_t} \ge \frac{{{p_{\min }}}}{{1 - neg_{th}}}$$ 
    - $${p_{\min }} \le {p_{\max }}\left( {1 - neg_{th}} \right)$$
    则认为 $L_t=0$，即极端看跌
  - 否则 
    - $${L_t} = 1 - \frac{{{P_t} - {P_{\min }}}}{{\frac{{\min }}{{1 - ne{g_{th}}}} - {P_{\min }}}}$$
    $p_t$ 越接近于 ${\frac{{\min }}{{1 - ne{g_{th}}}}}$，$L_t$ 越接近于 0，即越看跌；否则 $p_t$ 越接近于 $p_{min}$，$L_t$ 越接近于 1，看跌减弱

## `PctChange`
百分比变化趋势标签模式，计算每个时间点到下一个极值点的百分比变化
    
标签含义：
- 正值：到下一个极值点的上涨百分比
- 负值：到下一个极值点的下跌百分比
- 0：价格无变化

计算公式：
$$label = \frac{{next\_extrema\_price{\rm{ }} - {\rm{ }}current\_price}}{{current\_price}}$$

## `PctChangeNorm`
标准化百分比变化趋势标签模式
    
标签含义：
- 正值：到下一个极值点的标准化上涨百分比
- 负值：到下一个极值点的标准化下跌百分比
- 0：价格无变化

计算公式：

对于上涨趋势 $$label = \frac{{next\_extrema\_price{\rm{ }} - {\rm{ }}current\_price}}{{next\_extrema\_price}}$$
对于下跌趋势
$$label = \frac{{next\_extrema\_price{\rm{ }} - {\rm{ }}current\_price}}{{current\_price}}$$

# TrendMode
`TrendModeT` 类型的全局单例实例，提供了所有趋势标签生成模式的访问接口。
```python
TrendMode = TrendModeT()
```